# Rossmann Store Sales — Data Understanding

## Objective

Understand the structure, quality, and basic characteristics of the raw Rossmann datasets before performing any cleaning, feature engineering, or modeling.

## Datasets

- `train.csv`: Historical daily sales data for Rossmann stores
- `store.csv`: Store-level metadata

## Goals

- Inspect dataset dimensions and schema
- Review data types
- Identify missing values
- Identify duplicated records
- Inspect categorical values
- Understand the date range and store coverage
- Detect potential data quality issues

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = RAW_DATA_DIR / "train.csv"
STORE_PATH = RAW_DATA_DIR / "store.csv"

## 2. Verify Data Paths

In [ ]:
print(f"Train exists: {TRAIN_PATH.exists()}")
print(f"Store exists: {STORE_PATH.exists()}")

## 3. Load Raw Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
store = pd.read_csv(STORE_PATH)

## 4. Initial Dataset Inspection

Inspect the size, sample rows, column names, and inferred data types before making any modifications to the raw data.


In [ ]:
train.shape

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.nunique()

In [ ]:
train["StateHoliday"].value_counts(dropna=False)

In [ ]:
train["StateHoliday"].unique()

In [ ]:
train["StateHoliday"].map(type).value_counts()

### Key Observations

- The training dataset contains 1,017,209 observations and 9 columns.
- Each observation represents one store on one date.
- The dataset covers 1,115 unique stores and 942 unique dates.
- `Sales` is the target variable.
- `Store` is an identifier rather than a continuous numerical variable.
- `Open`, `Promo`, and `SchoolHoliday` are binary indicators.
- `DayOfWeek` is numerically encoded but represents a calendar category.
- `Date` is currently loaded as a string and should later be treated as a temporal variable.
- `StateHoliday` contains inconsistent representations of the no-holiday category: both integer `0` and string `"0"` are present. Semantically, the column represents four categories (`0`, `a`, `b`, `c`) and will require type standardization during data cleaning.
- `Customers` is numerical, but its availability at prediction time must be considered before using it as a model feature.

### 4.2 Data Quality Checks

Check for missing values, duplicated rows, and duplicate store-date combinations before performing any cleaning.

In [ ]:
train.isna().sum()

In [ ]:
train.duplicated().sum()

In [ ]:
train.duplicated(subset=["Store", "Date"]).sum()

### Key Findings

- No missing values were detected in the training dataset.
- No fully duplicated rows were found.
- The combination of `Store` and `Date` is unique across the dataset.
- This confirms that each observation represents exactly one store on one date.
- Although no missing values are present, data quality issues may still exist, as observed in the inconsistent representation of `StateHoliday`.

### 4.3 Temporal Coverage

Inspect the time span, date ordering, and store-level coverage of the training data without modifying the raw dataset.

In [ ]:
date_parsed = pd.to_datetime(train["Date"], errors="raise")

In [ ]:
date_parsed.min(), date_parsed.max(), date_parsed.nunique()

In [ ]:
expected_days = (date_parsed.max() - date_parsed.min()).days + 1

In [ ]:
date_parsed.nunique()

In [ ]:
expected_days

- The training data spans from `2013-01-01` to `2015-07-31`.
- The dataset contains 942 unique dates, equal to the total number of calendar days in this interval.
- Therefore, there are no globally missing calendar dates in the training period.
- This does not imply that every store has observations for all 942 dates; store-level coverage must be checked separately.

In [ ]:
date_parsed.is_monotonic_increasing

In [ ]:
date_parsed.is_monotonic_decreasing

In [ ]:
store_date_counts = train.groupby("Store")["Date"].nunique()
store_date_counts.describe()

In [ ]:
store_date_counts.sort_values().head(10)

In [ ]:
store_date_counts.value_counts().sort_index()

### 4.4 Store-Level Time Span Consistency

Compare the observed number of dates for each store with the expected number of calendar days between its first and last observation.

In [ ]:
store_coverage = (
    train.assign(DateParsed=date_parsed)
    .groupby("Store")
    .agg(
        first_date=("DateParsed", "min"),
        last_date=("DateParsed", "max"),
        observed_days=("DateParsed", "nunique"),
    )
)

In [ ]:
store_coverage.head()

In [ ]:
store_coverage["expected_span_days"] = (
    store_coverage["last_date"] - store_coverage["first_date"]
).dt.days + 1

In [ ]:
store_coverage["missing_within_span"] = (
    store_coverage["expected_span_days"] - store_coverage["observed_days"]
)

In [ ]:
store_coverage["missing_within_span"].value_counts().sort_index()

In [ ]:
store_485_dates = pd.DatetimeIndex(
    train.loc[train["Store"] == 485, "Date"].pipe(pd.to_datetime)
)

In [ ]:
expected_485_dates = pd.date_range(
    start=store_485_dates.min(),
    end=store_485_dates.max(),
    freq="D",
)

In [ ]:
missing_485_dates = expected_485_dates.difference(store_485_dates)

In [ ]:
len(missing_485_dates)

In [ ]:
missing_485_dates.min(), missing_485_dates.max()

In [ ]:
missing_485_dates[:10]

### Finding: Store 485 Missing Interval

Store 485 has 184 missing dates within its observed time span.

The missing dates form one continuous interval from `2014-07-01` to `2014-12-31`, rather than being randomly scattered throughout the dataset.

This suggests a structured gap in the historical records and motivates checking whether the other affected stores share the same missing interval.

In [ ]:
affected_stores = store_coverage.index[store_coverage["missing_within_span"] == 184]

In [ ]:
len(affected_stores)

In [ ]:
all_dates = pd.date_range(
    start=date_parsed.min(),
    end=date_parsed.max(),
    freq="D",
)

In [ ]:
affected_presence = (
    train.assign(DateParsed=date_parsed)
    .loc[lambda df: df["Store"].isin(affected_stores)]
    .groupby("DateParsed")["Store"]
    .nunique()
    .reindex(all_dates, fill_value=0)
)

In [ ]:
affected_presence.value_counts().sort_index()

In [ ]:
missing_dates_all_affected = affected_presence[affected_presence == 0].index

missing_dates_all_affected.min(), missing_dates_all_affected.max()

### Finding: Shared Six-Month Gap Across 180 Stores

A group of 180 stores has a structured temporal gap.

- All 180 affected stores are present on 758 dates.
- None of the 180 stores are present on 184 dates.
- The missing interval is continuous and spans from `2014-07-01` through `2014-12-31`.
- This pattern is systematic rather than randomly scattered across stores or dates.
- The absence represents missing records rather than conventional null values, since these store-date observations do not exist in the dataset at all.

The cause of this gap should not be assumed without additional evidence. It may reflect a business or data-collection process and should be considered later when designing features and validation.

In [ ]:
store_date_counts[store_date_counts == 941]

In [ ]:
store_941_id = store_date_counts[store_date_counts == 941].index[0]

store_941_id

In [ ]:
store_coverage.loc[store_941_id]

### Temporal Coverage Summary

- The training dataset spans from `2013-01-01` to `2015-07-31`, covering 942 consecutive calendar days.
- 934 stores contain observations for all 942 dates.
- 180 stores contain only 758 observations and share the same continuous 184-day gap from `2014-07-01` through `2014-12-31`.
- Store `988` contains 941 observations. Its first available record is `2013-01-02`, and it has no missing dates between its first and last observations.
- Therefore, reduced observation counts do not always indicate the same type of missing-data problem. Store-level temporal coverage must be interpreted relative to each store's observed time span.

In [ ]:
expected_day_of_week = date_parsed.dt.dayofweek + 1

In [ ]:
(train["DayOfWeek"] != expected_day_of_week).sum()

- `DayOfWeek` is fully consistent with the calendar date across all training observations; no mismatches were detected.

## 5. Store Metadata Inspection

Inspect the structure and data quality of `store.csv` before merging it with the training data.

In [ ]:
store.shape

In [ ]:
store.head()

In [ ]:
store.info()

In [ ]:
store.nunique()

In [ ]:
store.isna().sum()

### 5.1 Store-Level Integrity Checks

Verify that each store appears exactly once and inspect whether missing promotional fields are structurally related to `Promo2`.

In [ ]:
store["Store"].nunique()

In [ ]:
store["Store"].duplicated().sum()

In [ ]:
store["Promo2"].value_counts()

In [ ]:
promo2_columns = ["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]

In [ ]:
store.loc[store["Promo2"] == 0, promo2_columns].isna().sum()

In [ ]:
store.loc[store["Promo2"] == 1, promo2_columns].isna().sum()

### Promo2 Missingness Finding

Missing values in `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval` are structurally related to `Promo2`.

- All 544 stores with `Promo2 = 0` have missing values in these three fields.
- All 571 stores with `Promo2 = 1` have complete Promo2 metadata.
- Therefore, these missing values represent "not applicable" information rather than unexpected missing data.

### 5.2 Competition Missingness

Investigate whether missing competition-related fields follow a structural pattern or represent genuinely unavailable information.

In [ ]:
competition_columns = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

In [ ]:
store[competition_columns].isna().value_counts()

In [ ]:
store.loc[
    store["CompetitionDistance"].isna(),
    ["Store", *competition_columns],
]

### Competition Missingness Findings

- 761 stores have complete competition information.
- 351 stores have a known `CompetitionDistance` but missing `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear`.
- 3 stores are missing all competition-related fields.
- `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` are always missing together.
- Unlike Promo2-related missing values, competition missingness cannot yet be treated as purely structural or "not applicable".

- Stores `291`, `622`, and `879` have no recorded competition information.
- For these stores, `CompetitionDistance`, `CompetitionOpenSinceMonth`, and `CompetitionOpenSinceYear` are all missing.
- The dataset alone does not tell us whether these stores have no nearby competitor or whether the competition information is simply unavailable.

### 5.3 Competition Value Validation

Validate the ranges and possible values of the competition-related variables before deciding how to handle missing values.

In [ ]:
store["CompetitionOpenSinceMonth"].value_counts(dropna=False).sort_index()

In [ ]:
store["CompetitionOpenSinceYear"].value_counts(dropna=False).sort_index()

In [ ]:
store["CompetitionDistance"].describe()

In [ ]:
store["CompetitionDistance"].sort_values().head(10)

In [ ]:
store.nlargest(
    10,
    "CompetitionDistance",
)[["Store", "CompetitionDistance"]]

In [ ]:
store.nsmallest(
    10,
    "CompetitionDistance",
)[["Store", "CompetitionDistance"]]

In [ ]:
store.loc[
    store["CompetitionOpenSinceYear"] <= 1990,
    [
        "Store",
        "CompetitionDistance",
        "CompetitionOpenSinceMonth",
        "CompetitionOpenSinceYear",
    ],
]

### Competition Value Findings

- `CompetitionOpenSinceMonth` contains only valid values from 1 to 12.
- `CompetitionDistance` ranges from 20 to 75,860 and is strongly right-skewed.
- Large competition distances are not automatically treated as invalid outliers.
- Store `815` has `CompetitionOpenSinceYear = 1900`, which is unusually old and should be flagged for later investigation rather than automatically corrected or removed.
- Other old competition years, such as 1961 and 1990, are possible and should not be treated as errors without additional evidence.

In [ ]:
store["StoreType"].value_counts(dropna=False)

In [ ]:
store["Assortment"].value_counts(dropna=False)

In [ ]:
store["PromoInterval"].value_counts(dropna=False)

### Store Metadata Summary

- `store.csv` contains one unique record for each of the 1,115 stores.
- `StoreType` contains four valid categories (`a`, `b`, `c`, `d`) with no missing values.
- `Assortment` contains three valid categories (`a`, `b`, `c`) with no missing values.
- Promo2-related missing values are structural:
  - 544 stores with `Promo2 = 0` have no Promo2 metadata.
  - All 571 stores with `Promo2 = 1` have complete Promo2 metadata.
- `PromoInterval` contains three valid promotional schedules for Promo2-participating stores.
- Competition-related missingness is different from Promo2 missingness and cannot be treated as purely structural.
- `CompetitionOpenSinceMonth` contains only valid month values from 1 to 12.
- `CompetitionDistance` is strongly right-skewed, but extreme values are not automatically considered invalid.
- Store `815` has an unusually old `CompetitionOpenSinceYear` value of 1900 and should be flagged for later investigation rather than automatically modified.

In [ ]:
train_store_ids = set(train["Store"])
metadata_store_ids = set(store["Store"])

train_store_ids == metadata_store_ids

In [ ]:
train_store_ids - metadata_store_ids

In [ ]:
metadata_store_ids - train_store_ids

## Final Data Understanding Summary

- `train.csv` and `store.csv` cover the same 1,115 stores.
- Each `Store`-`Date` combination in the training data is unique.
- The training data contains no standard null values, but it includes structured temporal gaps and an inconsistent `StateHoliday` representation.
- Store metadata contains structural missingness related to `Promo2` and unresolved missingness in competition-related fields.
- Store-level metadata, categorical values, and competition-related ranges were inspected before cleaning.
- No cleaning or feature engineering has been applied yet.